# 📖 Lab 3: Politeness — robots.txt + Domain Rate Limiting

**Non-functional requirement:** *Respect robots.txt and don't overload servers.*

Without politeness, our crawler could get banned, overload servers, or violate website policies. We need:
1. **robots.txt** — check which pages are allowed + respect crawl delays
2. **Domain rate limiting** — max 1 request/sec per domain across all crawlers

## Learning Objectives

- Parse real robots.txt files and check URL permissions
- Implement per-domain rate limiting with Redis
- Add jitter to prevent thundering herd
- See how crawl delay affects throughput

In [ ]:
import requests
import redis
import time
import random
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser

redis_client = redis.Redis(host="localhost", port=6382, decode_responses=True)
redis_client.flushdb()
print(f"✅ Redis: connected")

## 🤖 robots.txt: Checking Permissions

Python has a built-in `RobotFileParser` that handles the Robots Exclusion Protocol. Let's use it to check real websites.

In [ ]:
robots_cache: dict[str, RobotFileParser] = {}

def get_robots(domain: str) -> RobotFileParser:
    """Fetch and cache robots.txt for a domain."""
    if domain in robots_cache:
        return robots_cache[domain]

    rp = RobotFileParser()
    robots_url = f"https://{domain}/robots.txt"
    try:
        rp.set_url(robots_url)
        rp.read()
        robots_cache[domain] = rp
        return rp
    except Exception:
        robots_cache[domain] = rp
        return rp


def is_allowed(url: str, user_agent: str = "*") -> dict:
    """Check if a URL is allowed by robots.txt."""
    parsed = urlparse(url)
    domain = parsed.netloc
    rp = get_robots(domain)

    allowed = rp.can_fetch(user_agent, url)
    crawl_delay = rp.crawl_delay(user_agent)

    return {
        "url": url,
        "domain": domain,
        "allowed": allowed,
        "crawl_delay": crawl_delay,
    }


# Test with real websites
test_urls = [
    "https://www.google.com/",
    "https://www.google.com/search?q=test",
    "https://en.wikipedia.org/wiki/Python",
    "https://twitter.com/",
    "https://example.com/",
]

print("🤖 robots.txt permission check:\n")
print(f"  {'URL':<45} {'Allowed':<10} {'Crawl-delay'}")
print(f"  {'─'*65}")

for url in test_urls:
    result = is_allowed(url)
    icon = "✅" if result["allowed"] else "❌"
    delay = result["crawl_delay"] if result["crawl_delay"] else "—"
    print(f"  {url:<45} {icon:<10} {delay}")

## 🔒 Domain Rate Limiting with Redis

Multiple crawlers must coordinate: max 1 request/sec per domain globally. We use Redis `SET NX EX` as an atomic per-domain lock with TTL.

In [ ]:
DEFAULT_CRAWL_DELAY = 1  # 1 second between requests to same domain

def acquire_domain_lock(domain: str, crawl_delay: int = None) -> bool:
    """
    Atomic per-domain lock using Redis SET NX EX.
    Returns True if lock acquired (OK to crawl), False if domain is rate-limited.
    """
    delay = crawl_delay or DEFAULT_CRAWL_DELAY
    lock_key = f"domain_lock:{domain}"
    acquired = redis_client.set(lock_key, "locked", nx=True, ex=delay)
    return bool(acquired)


def polite_fetch(url: str) -> dict:
    """
    Complete polite fetch: check robots.txt → acquire domain lock → fetch.
    """
    parsed = urlparse(url)
    domain = parsed.netloc

    # Step 1: Check robots.txt
    robot_check = is_allowed(url)
    if not robot_check["allowed"]:
        return {"url": url, "status": "disallowed_by_robots", "fetched": False}

    crawl_delay = robot_check["crawl_delay"] or DEFAULT_CRAWL_DELAY

    # Step 2: Acquire domain lock (rate limiting)
    if not acquire_domain_lock(domain, int(crawl_delay)):
        return {"url": url, "status": "rate_limited", "fetched": False,
                "message": f"Domain {domain} was crawled recently. Wait {crawl_delay}s."}

    # Step 3: Fetch
    try:
        resp = requests.get(url, timeout=5, headers={"User-Agent": "EducationalCrawlerBot/1.0"})
        resp.raise_for_status()
        return {"url": url, "status": "success", "fetched": True, "size": len(resp.text)}
    except requests.RequestException as e:
        return {"url": url, "status": "error", "fetched": False, "error": str(e)}


# Test: rapid requests to the same domain
redis_client.flushdb()

print("🔒 Domain rate limiting demo:\n")
print("  Sending 3 rapid requests to example.com:\n")

for i in range(3):
    result = polite_fetch("https://example.com/")
    if result["fetched"]:
        print(f"  Request {i+1}: ✅ {result['status']} ({result['size']} bytes)")
    else:
        print(f"  Request {i+1}: ❌ {result['status']} — {result.get('message', '')}")

print(f"\n  💡 Only the first request went through.")
print(f"     The domain lock (TTL=1s) prevents hammering the same server.")

## 🎲 Jitter: Preventing Thundering Herd

Without jitter, all crawlers waiting for the same domain retry at the exact same time when the lock expires. Only one wins, the rest wait again. **Jitter** adds random delay so retries spread out.

In [ ]:
import threading

redis_client.flushdb()

def crawler_with_jitter(crawler_id: int, domain: str, results: list):
    """Simulates a crawler that retries with jitter on rate limit."""
    max_attempts = 5
    for attempt in range(1, max_attempts + 1):
        if acquire_domain_lock(domain, 1):
            results.append({"crawler": crawler_id, "attempt": attempt, "status": "fetched"})
            return
        # Jitter: random 0.1-0.5s on top of the 1s delay
        jitter = random.uniform(0.1, 0.5)
        time.sleep(jitter)

    results.append({"crawler": crawler_id, "attempt": max_attempts, "status": "gave_up"})

# Simulate 5 crawlers all trying to fetch from the same domain
print("🎲 5 crawlers competing for 'example.com' with jitter:\n")

results = []
threads = []
for i in range(5):
    t = threading.Thread(target=crawler_with_jitter, args=(i + 1, "example.com", results))
    threads.append(t)

for t in threads:
    t.start()
for t in threads:
    t.join()

results.sort(key=lambda r: r["crawler"])
for r in results:
    icon = "✅" if r["status"] == "fetched" else "⏳"
    print(f"  Crawler {r['crawler']}: {icon} {r['status']} (attempt {r['attempt']})")

fetched = sum(1 for r in results if r["status"] == "fetched")
print(f"\n  {fetched}/5 crawlers got through — jitter spread out the retries.")
print(f"  Without jitter, all 5 would retry at the exact same time → only 1 wins each round.")

## 🧹 Cleanup

In [ ]:
redis_client.flushdb()
print("✅ Redis cleaned up.")

## ✅ Summary

| Mechanism | Implementation | Purpose |
|-----------|---------------|---------|
| **robots.txt** | `RobotFileParser` — check `Disallow` + `Crawl-delay` | Respect site policies |
| **Domain lock** | Redis `SET NX EX` (1s TTL per domain) | Max 1 req/sec/domain |
| **Jitter** | Random 0.1-0.5s delay on retry | Prevent thundering herd |
| **Crawl-delay** | Override lock TTL with robots.txt value | Honor site-specific delays |

**Next:** Lab 4 — Efficiency (URL + content deduplication, crawler traps)